#Installtion & setup

In [ ]:
!pip install -qU langchain langchain-core langchain-google-genai langchain-community

In [27]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

model.invoke("Hi").content

# Prompt

### 1. Uses of the PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="You are a helful assistance you answer the following question {question} in to 5 importance answer asked by the user",
    input_variables=["question"]
)

prompt.save("template.json")
userQuery = prompt.format(question="What is the capital of India?")
model.invoke(userQuery).content

"The capital of India is **New Delhi**.\n\nHere are 5 important aspects about New Delhi:\n\n1.  **National Capital & Seat of Government:** New Delhi serves as the political and administrative center of India. It houses the Rashtrapati Bhavan (President's Residence), Parliament House, Supreme Court, and numerous government ministries, making it the heart of the nation's governance.\n2.  **Rich Historical and Cultural Heritage:** The city is steeped in history, with a fascinating blend of ancient Mughal architecture and colonial-era structures. It boasts iconic landmarks like India Gate, Humayun's Tomb, Qutub Minar, Red Fort, and Lotus Temple, reflecting centuries of diverse cultural influences.\n3.  **Economic and Commercial Hub:** Beyond its political role, New Delhi is a significant economic center. It's a major hub for trade, finance, education, and technology, attracting businesses and talent from across the country and the world.\n4.  **Modern Metropolis with Robust Infrastructure:

### 2. ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    [
        ("system","You are a helful doctor assitances"),
        ("ai"," which provide all the prescribtion that is essiential in any suffering and if the issue is critical suggest the user for the hospital visit or treatment."),
        ("human","{user_input}")
    ]
)
prompt_value = template.invoke(
   { "user_input":"I have a cough"}
)
model.invoke(prompt_value).content

"I'm sorry to hear you have a cough. To help me understand better and offer the most appropriate suggestions, could you please tell me a bit more about it?\n\n1.  **How long have you had the cough?**\n2.  **Is it a dry cough or are you bringing up phlegm?** If phlegm, what color is it?\n3.  **Do you have any other symptoms** like fever, sore throat, runny nose, body aches, headache, chest pain, or shortness of breath?\n4.  **Is it worse at certain times** (e.g., night, morning, after eating)?\n5.  **Are you experiencing any difficulty breathing, wheezing, or tightness in your chest?**\n6.  **Do you have any known allergies or underlying medical conditions** (like asthma, COPD, heart conditions)?\n\nIn the meantime, here are some general recommendations that can often help with a cough, depending on its nature:\n\n**General Recommendations for Cough Relief:**\n\n*   **Stay Hydrated:** Drink plenty of fluids like water, herbal teas, or warm broths. This helps to soothe your throat and th

# Structure Output

## 1. with_structure_output

### typedDict

In [ ]:
# simple typedDict

from typing import TypedDict

# schema
class Review(TypedDict):
  summary: str
  sentiment: str


structure_model = model.with_structured_output(Review)
result = structure_model.invoke("I love s23 ultra")

print(result['sentiment'])
print(result['summary'])

Positive
The user loves the s23 ultra


In [ ]:
# Annoted typedDict

from typing import TypedDict, Annotated

# schema
class Review(TypedDict):
  summary: Annotated[str, "A breif summary of the review"]
  sentiment: Annotated[int, "The sentiment of the review either 1 for postive or 0 for negative"]


structure_model = model.with_structured_output(Review)
result = structure_model.invoke("I love s23 ultra")

print(result['sentiment'])
print(result['summary'])

1
I love s23 ultra


In [ ]:
from typing import TypedDict, Annotated, Optional

class Review(TypedDict):

  Key_themes : Annotated[list[str], "Write down all the key theme discussed in the review in a list"]
  summary : Annotated[str, "A breif summary of the review"]
  sentiment: Annotated[str, "Return sentiment of the review either negative , positive or neutral"]
  pros: Annotated[Optional[list[str]], "Write down all the pros in a list"]
  cons: Annotated[Optional[list[str]], "Write down all the cons in a list"]
  name: Annotated[Optional[str], "Write the name of the reviewer"]


structure_model = model.with_structured_output(Review)

result = structure_model.invoke(""" I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Tushar Gupta""")

print(result)
print(result['summary'])
print(result['name'])

{'sentiment': 'Positive', 'pros': ['Insanely powerful Snapdragon 8 Gen 3 processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities (up to 30x effective zoom, excellent night mode)', 'Long 5000mAh battery life with 45W fast charging', 'S-Pen support is unique and useful'], 'cons': ['Weight and size make it uncomfortable for one-handed use', 'Samsung’s One UI still comes with bloatware', 'High $1,300 price tag', 'Zoom beyond 30x loses quality'], 'Key_themes': ['Performance', 'Camera', 'Battery Life', 'S-Pen', 'Design', 'Software', 'Price'], 'summary': "The Samsung Galaxy S24 Ultra offers top-tier performance with its Snapdragon 8 Gen 3 processor, an impressive 200MP camera with excellent night mode and zoom up to 30x, and a long-lasting 5000mAh battery with 45W fast charging. The S-Pen adds a useful dimension. However, its large size and weight make one-handed use difficult, Samsung's One UI includes bloatware, and the $1,300 price tag is a 

### pydantic

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

class Review(BaseModel):
  summary:str = Field(description="A brief summary of the review")
  sentiment:Literal["pos", "neg"] = Field(description="give me the sentiment of the review either positive or negative")
  pros:Optional[list[str]] = Field(description="Write down all the pros in a list")
  cons:Optional[list[str]] = Field(description="Write down all the cons in a list")
  reviver:Optional[str] = Field(description="Write the name of the reviewer")


structure_model = model.with_structured_output(Review)
result = structure_model.invoke(""" I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Tushar Gupta""")

print(result)
print(result.summary)
print(result.sentiment)
print(result.pros)
print(result.cons)
print(result.reviver)

summary='The Samsung Galaxy S24 Ultra is a powerful phone with a great camera, long battery life, and useful S-Pen, though its size, bloatware, and high price are drawbacks.' sentiment='pos' pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'] cons=['Heavy and large, uncomfortable for one-handed use', 'Samsung’s One UI includes bloatware', 'High price tag ($1,300)', 'Camera zoom beyond 30x loses quality'] reviver='Tushar Gupta'
The Samsung Galaxy S24 Ultra is a powerful phone with a great camera, long battery life, and useful S-Pen, though its size, bloatware, and high price are drawbacks.
pos
['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful']
['Heavy and large, uncomfortable for 

### json

In [ ]:
# siply convert the following pydantic in to json
json_data = result.model_dump()

print(json_data)
print(json_data['sentiment'])

{'summary': 'The Samsung Galaxy S24 Ultra is a powerful phone with a great camera, long battery life, and useful S-Pen, though its size, bloatware, and high price are drawbacks.', 'sentiment': 'pos', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'cons': ['Heavy and large, uncomfortable for one-handed use', 'Samsung’s One UI includes bloatware', 'High price tag ($1,300)', 'Camera zoom beyond 30x loses quality'], 'reviver': 'Tushar Gupta'}
pos
